# 02 Model Convert

Convert `best_line_follower_model_xy.onnx` to Horizon `.bin` from the current project directory.

This notebook is adjusted for Jupyter running under `/data`, and uses the imported Docker/OpenExplorer conversion idea so `hb_mapper` does not need to be installed in the current shell PATH.


In [ ]:
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path


def list_missing_files(base_dir: Path, required):
    return [name for name in required if not (base_dir / name).exists()]


def detect_project_dir():
    signature_files = [
        "prepare_horizon_mapper.py",
        "resnet18_224x224_nv12.yaml",
    ]
    helper_files = [
        "prepare_horizon_mapper.py",
        "horizon_preprocess.py",
        "02_preprocess.sh",
        "03_build.sh",
        "resnet18_224x224_nv12.yaml",
    ]
    seen = set()
    candidates = []

    def add_candidate(path):
        try:
            resolved = path.expanduser().resolve()
        except Exception:
            resolved = path.expanduser()
        key = str(resolved)
        if key not in seen:
            seen.add(key)
            candidates.append(resolved)

    env_project = os.environ.get("PROJECT_DIR")
    if env_project:
        add_candidate(Path(env_project))

    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        add_candidate(base)
    for extra in [
        cwd / "originbot",
        cwd.parent / "originbot",
        Path("/data/originbot"),
        Path("/root/originbot"),
        Path("/home/ubuntu/originbot"),
    ]:
        add_candidate(extra)

    for root in [cwd, Path("/data"), Path("/root"), Path("/home/ubuntu")]:
        if not root.is_dir():
            continue
        try:
            for marker in root.rglob("prepare_horizon_mapper.py"):
                add_candidate(marker.parent)
        except Exception:
            pass

    best_candidate = None
    best_score = -1
    diagnostics = []
    for candidate in candidates:
        if not candidate.is_dir():
            continue
        score = sum((candidate / name).exists() for name in helper_files)
        missing = list_missing_files(candidate, helper_files)
        diagnostics.append((score, candidate, missing))
        if all((candidate / name).exists() for name in signature_files):
            return candidate.resolve()
        if score > best_score:
            best_score = score
            best_candidate = candidate

    print("cwd:", cwd)
    if env_project:
        print("PROJECT_DIR:", env_project)
    if diagnostics:
        print("Project-dir candidates checked:")
        for score, candidate, missing in sorted(diagnostics, key=lambda x: (-x[0], str(x[1])))[:12]:
            print(f" - {candidate} | matched={score}/{len(helper_files)} | missing={missing}")

    if best_candidate is not None and best_score > 0:
        print(f"[WARN] Using best-effort project_dir candidate: {best_candidate}")
        return best_candidate.resolve()

    raise FileNotFoundError(
        "Project directory not found. Set PROJECT_DIR to the folder containing prepare_horizon_mapper.py, "
        "horizon_preprocess.py, 02_preprocess.sh, 03_build.sh and resnet18_224x224_nv12.yaml."
    )


def detect_oe_root():
    suffix_parts = ["samples", "ai_toolchain", "horizon_model_convert_sample", "03_classification"]
    seen = set()
    candidates = []

    def add_candidate(path):
        try:
            resolved = path.expanduser().resolve()
        except Exception:
            resolved = path.expanduser()
        key = str(resolved)
        if key not in seen:
            seen.add(key)
            candidates.append(resolved)

    def normalize_oe_candidate(path):
        try:
            resolved = path.expanduser().resolve()
        except Exception:
            resolved = path.expanduser()
        parts = list(resolved.parts)
        suffix_len = len(suffix_parts)
        for idx in range(len(parts) - suffix_len + 1):
            if parts[idx:idx + suffix_len] == suffix_parts:
                return Path(*parts[:idx]) if idx > 0 else Path(resolved.anchor)
        return resolved

    env_candidates = []
    for env_name in ("OPEN_EXPLORER_ROOT", "OE_ROOT"):
        value = os.environ.get(env_name)
        if value:
            env_candidates.append((env_name, value))
            add_candidate(normalize_oe_candidate(Path(value)))

    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        add_candidate(normalize_oe_candidate(base))

    search_roots = [Path("/data"), Path("/root"), Path("/home/ubuntu")]
    for root in search_roots:
        if not root.is_dir():
            continue
        try:
            for marker in root.rglob("03_classification"):
                if marker.is_dir() and marker.parent.name == "horizon_model_convert_sample":
                    add_candidate(normalize_oe_candidate(marker))
        except Exception:
            pass
        try:
            for child in root.iterdir():
                if child.is_dir() and "horizon_x5_open_explorer" in child.name:
                    add_candidate(normalize_oe_candidate(child))
        except Exception:
            pass

    diagnostics = []
    for candidate in candidates:
        if not candidate.is_dir():
            continue
        sample_root = candidate / "samples/ai_toolchain/horizon_model_convert_sample/03_classification"
        exists = sample_root.is_dir()
        diagnostics.append((exists, candidate, sample_root))
        if exists:
            return candidate.resolve()
    for env_name, value in env_candidates:
        candidate = normalize_oe_candidate(Path(value))
        if candidate.is_dir():
            print(f"[WARN] Standard sample root not found under {candidate}. Using it as oe_root anyway.")
            return candidate.resolve()

    print("cwd:", cwd)
    if env_candidates:
        for env_name, value in env_candidates:
            print(f"{env_name}: {value}")
    if diagnostics:
        print("OpenExplorer candidates checked:")
        for exists, candidate, sample_root in diagnostics[:12]:
            print(f" - {candidate} | sample_root_exists={exists} | expected={sample_root}")

    raise FileNotFoundError(
        "OpenExplorer root not found. Set OPEN_EXPLORER_ROOT or OE_ROOT explicitly."
    )


def resolve_work_root(oe_root: Path):
    env_work_root = os.environ.get("OPEN_EXPLORER_WORK_ROOT") or os.environ.get("OE_WORK_ROOT") or os.environ.get("SAMPLE_ROOT")
    if env_work_root:
        work_root = Path(env_work_root).expanduser().resolve()
        work_root.mkdir(parents=True, exist_ok=True)
        return work_root

    standard_root = oe_root / "samples/ai_toolchain/horizon_model_convert_sample/03_classification"
    if standard_root.is_dir():
        return standard_root.resolve()

    fallback_root = oe_root / "originbot_convert_workspace"
    fallback_root.mkdir(parents=True, exist_ok=True)
    print(f"[WARN] Using fallback work_root because standard sample root is missing: {fallback_root}")
    return fallback_root.resolve()


def to_container_path(host_path: Path, mount_root: Path, container_root: Path = Path("/open_explorer")):
    relative = host_path.resolve().relative_to(mount_root.resolve())
    return container_root / relative


def run(cmd: str):
    print(f"\n$ {cmd}\n")
    completed = subprocess.run(
        cmd,
        shell=True,
        executable="/bin/bash",
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")
    return completed


project_dir = detect_project_dir()
oe_root = detect_oe_root()
work_root = resolve_work_root(oe_root)
work_name = os.environ.get("WORK_NAME", "originbot_convert")
mapper_dir = work_root / work_name / "mapper"
container_mapper_dir = to_container_path(mapper_dir, oe_root)
dataset_dir = Path(os.environ.get("CAL_DATASET_DIR", str(project_dir / "image_dataset"))).resolve()
onnx_path = Path(os.environ.get("ONNX_PATH", str(project_dir / "best_line_follower_model_xy.onnx"))).resolve()
docker_image = os.environ.get("DOCKER_IMAGE", "openexplorer/ai_toolchain_ubuntu_20_x5_cpu:v1.2.8-py310")
docker_cmd = os.environ.get("DOCKER_CMD", "docker")
march = os.environ.get("MARCH", "bayes-e")
docker_bin = docker_cmd.split()[0]

required_project_files = [
    "prepare_horizon_mapper.py",
    "horizon_preprocess.py",
    "02_preprocess.sh",
    "03_build.sh",
    "resnet18_224x224_nv12.yaml",
]
missing = [name for name in required_project_files if not (project_dir / name).exists()]
if missing:
    raise FileNotFoundError("Missing files in project_dir:\n" + "\n".join(f" - {name}" for name in missing))
if not onnx_path.is_file():
    raise FileNotFoundError(f"ONNX file not found: {onnx_path}")
if not dataset_dir.is_dir():
    raise FileNotFoundError(f"Calibration dataset not found: {dataset_dir}")
if shutil.which(docker_bin) is None:
    raise RuntimeError(f"Docker command not found: {docker_cmd}")

print("project_dir:", project_dir)
print("oe_root:", oe_root)
print("work_root:", work_root)
print("mapper_dir:", mapper_dir)
print("container_mapper_dir:", container_mapper_dir)
print("dataset_dir:", dataset_dir)
print("onnx_path:", onnx_path)
print("docker_cmd:", docker_cmd)
print("docker_image:", docker_image)
print("march:", march)
python_bin = sys.executable


In [ ]:
helper_files = [
    "prepare_horizon_mapper.py",
    "horizon_preprocess.py",
    "02_preprocess.sh",
    "03_build.sh",
    "run_horizon_convert.sh",
    "resnet18_224x224_nv12.yaml",
]

mapper_dir.mkdir(parents=True, exist_ok=True)
for name in helper_files:
    src = project_dir / name
    if not src.exists():
        continue
    dst = mapper_dir / name
    shutil.copy2(src, dst)
    if dst.suffix == ".sh":
        dst.chmod(0o755)

prepare_cmd = [
    python_bin, str(mapper_dir / "prepare_horizon_mapper.py"),
    "--mapper-dir", str(mapper_dir),
    "--onnx", str(onnx_path),
    "--dataset-dir", str(dataset_dir),
]
print("RUN:", " ".join(shlex.quote(part) for part in prepare_cmd))
subprocess.run(prepare_cmd, check=True)

jpg_count = len(list((mapper_dir / "image_dataset").glob("*.jpg")))
print("Mapper ready:", mapper_dir)
print("Mapper image count:", jpg_count)


In [ ]:
inner_cmd = (
    f"cd {shlex.quote(str(container_mapper_dir))} && "
    "rm -rf calibration_data_bgr_f32 && "
    "bash 02_preprocess.sh && "
    f"hb_mapper checker --model-type onnx --model ./best_line_follower_model_xy.onnx --march {shlex.quote(march)}"
)
docker_run_cmd = (
    f"{docker_cmd} run --rm "
    f"-v {shlex.quote(str(oe_root))}:/open_explorer "
    f"{docker_image} bash -lc {shlex.quote(inner_cmd)}"
)
run(docker_run_cmd)
print("calibration files =", len(list((mapper_dir / "calibration_data_bgr_f32").glob("*.rgb"))))


In [ ]:
inner_cmd = f"cd {shlex.quote(str(container_mapper_dir))} && bash 03_build.sh"
docker_run_cmd = (
    f"{docker_cmd} run --rm "
    f"-v {shlex.quote(str(oe_root))}:/open_explorer "
    f"{docker_image} bash -lc {shlex.quote(inner_cmd)}"
)
run(docker_run_cmd)

output_dir = mapper_dir / "model_output"
if not output_dir.exists():
    raise FileNotFoundError(f"model_output not found: {output_dir}")

for path in sorted(output_dir.iterdir()):
    print(" -", path.name, f"({path.stat().st_size / 1024 / 1024:.2f} MB)")

bin_candidates = sorted(output_dir.glob("*.bin"))
if not bin_candidates:
    raise RuntimeError(f"No .bin generated in {output_dir}")

latest_bin = bin_candidates[0]
easy_bin = project_dir / "model_latest.bin"
shutil.copy2(latest_bin, easy_bin)
print("copied from:", latest_bin)
print("copied to  :", easy_bin)
